In [165]:
from langchain_core.documents import Document

In [166]:
doc = Document(
    page_content="This is my text.",
    metadata={
        "author": "Rahul",
        "source": "sample.txt", 
        "date": "2026-05-24",
        "pages": 1
    }
)

doc

Document(metadata={'author': 'Rahul', 'source': 'sample.txt', 'date': '2026-05-24', 'pages': 1}, page_content='This is my text.')

In [167]:
import os
os.makedirs("../data/text_files", exist_ok=True)

from pathlib import Path
Path("../data/text_files/python_intro.txt").touch()
Path("../data/text_files/machine_learning_intro.txt").touch()

In [168]:
sample_texts = {
    "../data/text_files/python_intro.txt": """
    Python is a high-level, interpreted programming language known for its simplicity and readability. It was created by Guido van Rossum and first released in 1991. Python emphasizes code readability, allowing programmers to express concepts in fewer lines of code compared to languages like C++ or Java. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python's extensive standard library and vibrant community make it a popular choice for web development, data science, artificial intelligence, and automation tasks.
    """,
    "../data/text_files/machine_learning_intro.txt": """
    Machine Learning is a subset of artificial intelligence that enables computers to learn and make decisions from data. Instead of being explicitly programmed for every task, machine learning algorithms identify patterns in data and use these patterns to make predictions or take actions. Common applications include image recognition, natural language processing, recommendation systems, and fraud detection. Popular machine learning frameworks and libraries include TensorFlow, PyTorch, scikit-learn, and Keras. The field continues to evolve rapidly, driving innovations in various industries.
    """
}

for filepath, content in sample_texts.items():
    with open(filepath, 'w', encoding="utf-8") as f:
        f.write(content)

print("Sample text files created successfully!")

Sample text files created successfully!


In [169]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
document = loader.load()
print(document)


[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content="\n    Python is a high-level, interpreted programming language known for its simplicity and readability. It was created by Guido van Rossum and first released in 1991. Python emphasizes code readability, allowing programmers to express concepts in fewer lines of code compared to languages like C++ or Java. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python's extensive standard library and vibrant community make it a popular choice for web development, data science, artificial intelligence, and automation tasks.\n    ")]


In [170]:
from langchain_community.document_loaders import DirectoryLoader

directory_loader = DirectoryLoader(
    "../data/text_files", 
    glob="*.txt",
    loader_cls = TextLoader,
    loader_kwargs = {"encoding": "utf-8"},
    show_progress = False
)

documents = directory_loader.load()
documents

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content="\n    Python is a high-level, interpreted programming language known for its simplicity and readability. It was created by Guido van Rossum and first released in 1991. Python emphasizes code readability, allowing programmers to express concepts in fewer lines of code compared to languages like C++ or Java. It supports multiple programming paradigms, including procedural, object-oriented, and functional programming. Python's extensive standard library and vibrant community make it a popular choice for web development, data science, artificial intelligence, and automation tasks.\n    "),
 Document(metadata={'source': '../data/text_files/machine_learning_intro.txt'}, page_content='\n    Machine Learning is a subset of artificial intelligence that enables computers to learn and make decisions from data. Instead of being explicitly programmed for every task, machine learning algorithms identify patterns in d

In [171]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

directory_loader = DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls = PyMuPDFLoader,
    show_progress = False
)

pdf_documents = directory_loader.load()
pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-25T11:56:47+00:00', 'source': '../data/pdf/objectdetection.pdf', 'file_path': '../data/pdf/objectdetection.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-05-25T11:56:47+00:00', 'trapped': '', 'modDate': "D:20260525115647+00'00'", 'creationDate': "D:20260525115647+00'00'", 'page': 0}, page_content='Object Detection in Computer Vision\nAbstract\nObject detection is a fundamental computer vision task that involves identifying and localizing objects\nwithin images. This paper reviews modern deep learning approaches including YOLO, Faster R-CNN,\nand transformer-based detectors, comparing accuracy and inference speed.\nRegion-Based Methods\nRegion-based convolutional neural networks (R-CNN) introduced a two-stage pipeline: first generate\nregion proposals using 

In [172]:
from langchain_community.document_loaders import UnstructuredExcelLoader

directory_loader = DirectoryLoader(
    "../data/",
    glob="**/*.xlsx",
    loader_cls = UnstructuredExcelLoader,
    show_progress = False
)

excel_documents = directory_loader.load()

for doc in excel_documents:
    print(doc.page_content)
    print(doc.metadata)
    print("---")

Order ID Date Salesperson Region Product Category Units Unit Price ($) Revenue ($) Cost ($) Profit ($) Margin (%) ORD-001 2024-01-05 Alice Johnson North Laptop Pro 15 Electronics 12 1299 ORD-002 2024-01-08 Bob Martinez South Office Chair Furniture 25 349 ORD-003 2024-01-12 Carol White East Wireless Mouse Accessories 80 45 ORD-004 2024-01-15 David Lee West Standing Desk Furniture 8 899 ORD-005 2024-01-19 Alice Johnson North Monitor 27" Electronics 18 549 ORD-006 2024-01-22 Eve Turner Central Keyboard MX Accessories 60 129 ORD-007 2024-02-03 Bob Martinez South Laptop Pro 15 Electronics 7 1299 ORD-008 2024-02-07 Carol White East Webcam HD Accessories 45 89 ORD-009 2024-02-14 David Lee West Office Chair Furniture 15 349 ORD-010 2024-02-18 Eve Turner Central Monitor 27" Electronics 22 549 ORD-011 2024-03-02 Alice Johnson North Standing Desk Furniture 5 899 ORD-012 2024-03-09 Bob Martinez South Wireless Mouse Accessories 100 45 ORD-013 2024-03-15 Carol White East Laptop Pro 15 Electronics 10

In [173]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [174]:
 class EmbeddingManager:
    """Handles document embedding generation usubg SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
            Initializae the embedding manager
            
            Args:
                model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self.load_model()

    def load_model(self):
        """Load the sentence transformer model"""
        try:
            print(f"Loading sentence transformer model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully!. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings

        Returns:
            Numpy array of embeddings
        """
        if not self.model:
            raise ValueError("Model not loaded. Call load_model() first.")

        try:
            print(f"Generating embeddings for {len(texts)} texts")
            embeddings = self.model.encode(texts, show_progress_bar = True)
            print(f"Generated embeddings shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise


In [175]:
embedding_manager = EmbeddingManager()
embedding_manager

Loading sentence transformer model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6938.86it/s]


Model loaded successfully!. Embedding dimension: 384


/var/folders/9z/14rbqnqs78lfw5qqkljsxy6m0000gn/T/ipykernel_9631/3841590586.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully!. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [176]:
class VectoreStore:
    """Managed document embeddings in a chromaDB vectore store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vectore_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self.initialize_store()

    def initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Delete old collection if it exists with wrong distance metric
            try:
                existing = self.client.get_collection(self.collection_name)
                space = existing.metadata.get("hnsw:space", "l2")
                if space != "cosine":
                    print(f"Deleting old collection (metric={space}), recreating with cosine...")
                    self.client.delete_collection(self.collection_name)
            except Exception:
                pass

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG",
                    "hnsw:space": "cosine"
                }
            )

            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Document], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (document, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            doc_metadata = dict(document.metadata)
            doc_metadata["doc_index"] = i
            doc_metadata["content_length"] = len(document.page_content)
            metadatas.append(doc_metadata)

            documents_text.append(document.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(ids)} documents to vector store")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store = VectoreStore()
vector_store

Deleting old collection (metric=l2), recreating with cosine...
Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [177]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: objectdetection.pdf
  ✓ Loaded 1 pages

Processing: attention.pdf
  ✓ Loaded 11 pages

Processing: emneddings.pdf
  ✓ Loaded 1 pages

Processing: proposal.pdf
  ✓ Loaded 1 pages

Total documents loaded: 14


In [178]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [179]:
chunks=split_documents(all_pdf_documents)
chunks

Split 14 documents into 49 chunks

Example chunk:
Content: Object Detection in Computer Vision
Abstract
Object detection is a fundamental computer vision task that involves identifying and localizing objects
within images. This paper reviews modern deep learn...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-25T11:56:47+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-25T11:56:47+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/objectdetection.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'objectdetection.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-25T11:56:47+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-25T11:56:47+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '../data/pdf/objectdetection.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'objectdetection.pdf', 'file_type': 'pdf'}, page_content='Object Detection in Computer Vision\nAbstract\nObject detection is a fundamental computer vision task that involves identifying and localizing objects\nwithin images. This paper reviews modern deep learning approaches including YOLO, Faster R-CNN,\nand transformer-based detectors, comparing accuracy and inference speed.\nRegion-Based Methods\nRegion-based convolutional neural networks (R-CNN) introduced a two-stage pipeline: first generate\nregion proposals using selective search, then classify each region using a CNN. Faster R-C

In [180]:
text = [document.page_content for document in chunks]
text

['Object Detection in Computer Vision\nAbstract\nObject detection is a fundamental computer vision task that involves identifying and localizing objects\nwithin images. This paper reviews modern deep learning approaches including YOLO, Faster R-CNN,\nand transformer-based detectors, comparing accuracy and inference speed.\nRegion-Based Methods\nRegion-based convolutional neural networks (R-CNN) introduced a two-stage pipeline: first generate\nregion proposals using selective search, then classify each region using a CNN. Faster R-CNN\nimproved this by introducing the Region Proposal Network (RPN).\nSingle-Stage Detectors\nYOLO (You Only Look Once) reformulates object detection as a single regression problem, dividing the\nimage into a grid and predicting bounding boxes and class probabilities directly. This enables real-time\ndetection at 45 frames per second.\nEvaluation Metrics\nDetection performance is measured using mean Average Precision (mAP) across different IoU',
 'detection at

In [181]:
embeddings = embedding_manager.generate_embeddings(text)
embeddings

Generating embeddings for 49 texts


Batches: 100%|██████████| 2/2 [00:06<00:00,  3.17s/it]

Generated embeddings shape: (49, 384)


array([[ 0.03494398,  0.02716864,  0.03211834, ...,  0.01175786,
        -0.04155533, -0.08994403],
       [ 0.08643827, -0.05994639, -0.00813373, ..., -0.02441402,
        -0.07410643, -0.09260868],
       [-0.0798602 , -0.12790853,  0.0155921 , ...,  0.06883759,
        -0.07168015, -0.04826477],
       ...,
       [-0.03204337, -0.08539394, -0.01118337, ...,  0.00896279,
         0.01506906, -0.01009839],
       [-0.00428164, -0.10732598, -0.02315038, ...,  0.0269785 ,
        -0.08833768, -0.02819895],
       [-0.00538273,  0.04088328, -0.04007817, ..., -0.06623814,
        -0.03462791, -0.02667087]], shape=(49, 384), dtype=float32)

In [182]:
vector_store.add_documents(chunks, embeddings)

Adding 49 documents to vector store...
Successfully added 49 documents to vector store


In [183]:
class RAGRetriever:
    """Retriever for RAG pipeline using ChromaDB vector store"""

    def __init__(self, vector_store: VectoreStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        results = self.vector_store.collection.query(
            query_embeddings=[query_embedding.tolist()],
            n_results=top_k,
            include=["documents", "metadatas", "distances"]
        )

        retrieved_docs = []

        if results["documents"] and results["documents"][0]:
            documents = results["documents"][0]
            metadatas = results["metadatas"][0]
            distances = results["distances"][0]
            ids = results["ids"][0]

            for i, (doc, meta, distance, doc_id) in enumerate(zip(documents, metadatas, distances, ids)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "content": doc,
                        "metadata": meta,
                        "similarity_score": similarity_score,
                        "distance": distance,
                        "rank": i + 1
                    })
                    print(f"  Retrieved: {doc_id}, Score: {similarity_score:.4f}")
                else:
                    print(f"  Skipped: {doc_id}, Score: {similarity_score:.4f} (below threshold)")

        return retrieved_docs

rag_retriever = RAGRetriever(vector_store, embedding_manager)
rag_retriever.retrieve("What is attention is all you need?", top_k=3, score_threshold=0.1)

Retrieving documents for query: 'What is attention is all you need?'
Top K: 3, Score Threshold: 0.1
Generating embeddings for 1 texts


Batches: 100%|██████████| 1/1 [00:00<00:00, 21.95it/s]

Generated embeddings shape: (1, 384)
  Retrieved: doc_eb8583c9_27, Score: 0.4465
  Retrieved: doc_f5c2c49d_23, Score: 0.4376
  Retrieved: doc_d34b65a6_9, Score: 0.4194


[{'id': 'doc_eb8583c9_27',
  'content': 'convolution is equal to the combination of a self-attention layer and a point-wise feed-forward layer,\nthe approach we take in our model.\nAs side beneﬁt, self-attention could yield more interpretable models. We inspect attention distributions\nfrom our models and present and discuss examples in the appendix. Not only do individual attention\nheads clearly learn to perform different tasks, many appear to exhibit behavior related to the syntactic\nand semantic structure of the sentences.\n5 Training\nThis section describes the training regime for our models.\n5.1 Training Data and Batching\nWe trained on the standard WMT 2014 English-German dataset consisting of about 4.5 million\nsentence pairs. Sentences were encoded using byte-pair encoding [ 3], which has a shared source-\ntarget vocabulary of about 37000 tokens. For English-French, we used the signiﬁcantly larger WMT\n2014 English-French dataset consisting of 36M sentences and split tokens 